# 06 Model Evaluation
Evaluate models using classification metrics and visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc, 
    precision_recall_curve, f1_score
)
import joblib

# Load test data
from sklearn.model_selection import train_test_split
X = np.load('../data/combined_features.npy')
y = np.load('../data/labels.npy')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Load models
lr = joblib.load('../models/logistic_regression.joblib')
rf = joblib.load('../models/random_forest.joblib')
gb = joblib.load('../models/gradient_boosting.joblib')
svm = joblib.load('../models/svm.joblib')

models = {'Logistic Regression': lr, 'Random Forest': rf, 'Gradient Boosting': gb, 'SVM': svm}
print('Models loaded successfully')

# Get predictions
predictions = {}
for name, model in models.items():
    predictions[name] = model.predict(X_test)

print('Predictions generated')

# Classification metrics for each model
sns.set_style('whitegrid')
metrics_data = []

for name, y_pred in predictions.items():
    acc = (y_pred == y_test).mean()
    f1 = f1_score(y_test, y_pred)
    
    report = classification_report(y_test, y_pred, output_dict=True)
    precision = report['weighted avg']['precision']
    recall = report['weighted avg']['recall']
    
    metrics_data.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

metrics_df = pd.DataFrame(metrics_data).sort_values('F1-Score', ascending=False)
print('\n=== Classification Metrics ===')
print(metrics_df.to_string(index=False))
# Confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.ravel()

for idx, (name, y_pred) in enumerate(predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx], cbar=False)
    axes[idx].set_title(f'{name}\nAccuracy: {(y_pred == y_test).mean():.3f}')
    axes[idx].set_ylabel('True')
    axes[idx].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

# ROC curves
fig, ax = plt.subplots(figsize=(10, 6))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc:.3f})', linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier', linewidth=1)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# Precision-Recall curves
fig, ax = plt.subplots(figsize=(10, 6))

for name, model in models.items():
    if hasattr(model, 'predict_proba'):
        y_proba = model.predict_proba(X_test)[:, 1]
        precision, recall, _ = precision_recall_curve(y_test, y_proba)
        ax.plot(recall, precision, label=name, linewidth=2)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves')
ax.legend(loc='best')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


# Save evaluation report
import json
from pathlib import Path

eval_report = {
    'metrics': metrics_df.to_dict('records'),
    'best_model': metrics_df.iloc[0]['Model'],
    'best_f1': float(metrics_df.iloc[0]['F1-Score'])
}

report_path = '../assets/reports/evaluation_report.json'
Path(report_path).parent.mkdir(parents=True, exist_ok=True)

with open(report_path, 'w') as f:
    json.dump(eval_report, f, indent=2)

print(f'\nEvaluation report saved to {report_path}')